# In-Text Analysis (Figure 5 Section): Transient Theta-Burst Prevalence by RPE Class (eBOSC)

| Field | Value |
|---|---|
| **Manuscript panels / stats** | Results text: 'we next computed whether the prevalence of transient theta bursts differed between RPE classes ... We found no difference between conditions in either HPC or ACC (t-test, all ps > 0.05)' |
| **Inputs** | `$MEMORYBANDIT_DATA_DIR/projects/guLab/Salman/EphysAnalyses/{subj}/neural/Day1/{feedback_start,cue_on}-epo.fif`, `learn_df_RWH_v2.csv`, `full_df_RWH_v2.csv` |
| **Outputs** | `results/bosc/tables/bosc_period_stats_summary.csv`, `figures/BOSC_theta_bursts/period_averaged_band_comparison_rpe_bins.pdf` |
| **Dependencies** | `membandit`, `LFPAnalysis` (`eBOSC_wrapper`; set `LFPANALYSIS_ROOT`), `neurodsp` |
| **Expected runtime** | ~20-40 minutes across 19 subjects (seconds when loading precomputed per-subject BOSC tables) |
| **Upstream steps** | `01_behavior/Fig1DEG_S1_TableS1_modeling_EMU.ipynb` |


In [ ]:
# membandit setup
import os
import sys
from pathlib import Path

# Ensure cleaned_repo (parent of membandit/) is on sys.path when running from notebooks/
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "membandit" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

# Optional: set MEMORYBANDIT_DATA_DIR to your data root (defaults to $MEMORYBANDIT_DATA_DIR)
# os.environ.setdefault("MEMORYBANDIT_DATA_DIR", "/path/to/data/root")

from membandit.plotting_style import set_plot_style
set_plot_style()


# BOSC Theta/Alpha RPE Analysis

This notebook computes BOSC-based oscillatory burst prevalence during `feedback_start` and `cue_on`, using `learn_df` for non-cue events, `combined_df` for `cue_on`, and the shared behavior-loading helpers from `mv_granger_utils.py`.

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from membandit import paths
lfp_repo = os.environ.get('LFPANALYSIS_ROOT', '')
if lfp_repo and lfp_repo not in sys.path:
    sys.path.append(lfp_repo)
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from membandit.extras.bosc import RPE_CATEGORY_ORDER, assign_exact_band, build_period_plot_df, build_time_plot_df, compute_period_stats, load_and_preprocess_behavior, load_group_bosc_outputs, plot_period_averaged_bosc, plot_time_resolved_bosc, run_subject_event_bosc

sns.set_context('talk')
sns.set_style('white')


In [ ]:
base_dir = paths.get_base_dir()
load_dir = paths.get_emu_behavior_dir()
ephys_root = paths.get_ephys_dir()
demo_file = paths.get_subjects_xlsx_path()

events = {
    'feedback_start': (0.0, 1.5),
    'cue_on': (0.0, 1.0),
}
rois = ['ACC', 'dmPFC', 'HPC']
bands = {
    'theta': (4.0, 8.0),
    'alpha': (8.0, 13.0),
}

roi_colors = {
    'HPC': np.array([0.44314, 0.2902, 1.0]),
    'ACC': np.array([0.7098, 0.27059, 0.68235]),
    'dmPFC': np.array([0.40784, 0.01961, 0.0]),
}
rpe_palette = {
    'negative': '#E74C3C',
    'neutral': '#95A5A6',
    'positive': '#27AE60',
}

group_output_root = f'{ephys_root}/Plots/bosc_theta_alpha_rpe'
figure_dir = os.path.join(paths.figure_output_root(), 'BOSC_theta_bursts')
table_dir = os.path.join(paths.results_root(), 'bosc', 'tables')
os.makedirs(figure_dir, exist_ok=True)
os.makedirs(table_dir, exist_ok=True)

run_bosc = True
overwrite_bosc = False
n_jobs_bosc = -1

cfg_eBOSC_template = {
    'F': np.linspace(1, 30, 44),
    'wavenumber': 6,
    'pad.tfr_s': 0.0,
    'pad.detection_s': 0.0,
    'pad.background_s': 0.0,
    'threshold.excludePeak': np.array([]),
    'threshold.duration': None,
    'threshold.percentile': 0.95,
    'postproc.use': 'no',
    'postproc.method': 'FWHM',
    'postproc.edgeOnly': 'yes',
    'postproc.effSignal': 'PT',
    'trial': [],
    'trial_background': [],
}

demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion == 1) & (demo_data.Minerva == 1)].copy()
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])].copy()
subjects = subj_df.MSSMCode.tolist()

print(f'n_subjects={len(subjects)}')
print(f'group_output_root={group_output_root}')


In [ ]:
learn_df, combined_df = load_and_preprocess_behavior(load_dir)

print(learn_df[['participant', 'rpe', 'rpe_cat']].head())
print(combined_df[['participant', 'condition', 'rpe', 'rpe_cat', 'trials_mem']].head())


In [ ]:
combined_df.rpe

In [ ]:
combined_df[combined_df.participant == subj_id]

In [ ]:
audit_rows = []
if run_bosc:
    for subj_id in tqdm(subjects, desc='Computing BOSC by subject', ascii=True):
        for event in events:
            audit_rows.append(
                run_subject_event_bosc(
                    subj_id=subj_id,
                    event=event,
                    learn_df=learn_df,
                    combined_df=combined_df,
                    rois_to_use=rois,
                    ephys_root=ephys_root,
                    cfg_template=cfg_eBOSC_template,
                    overwrite=overwrite_bosc,
                    n_jobs=n_jobs_bosc,
                )
            )
else:
    print('Skipping BOSC computation and using existing subject-level outputs.')

audit_compute_df = pd.DataFrame(audit_rows)
if len(audit_compute_df) > 0:
    audit_compute_df.to_csv(f'{table_dir}/bosc_compute_audit.csv', index=False)
audit_compute_df.head()


In [ ]:
events = {
    'feedback_start': (0.0, 1.5),
    'cue_on': (0.0, 1.0),
}

In [ ]:
time_resolved_df, period_df, freq_df = load_group_bosc_outputs(
    subjects_to_use=subjects,
    rois_to_use=rois,
    events_to_use=events,
    ephys_root=ephys_root,
)

time_plot_df = build_time_plot_df(time_resolved_df)
period_plot_df = build_period_plot_df(period_df)
stats_long_df, stats_summary_df = compute_period_stats(
    period_df=period_df,
    rois=rois,
    events=events,
    bands=bands,
)

time_resolved_df.to_csv(f'{table_dir}/bosc_time_resolved_exact_band_all.csv', index=False)
period_df.to_csv(f'{table_dir}/bosc_period_averaged_exact_band_all.csv', index=False)
freq_df.to_csv(f'{table_dir}/bosc_time_averaged_frequency_all.csv', index=False)
time_plot_df.to_csv(f'{table_dir}/bosc_time_plot_df.csv', index=False)
period_plot_df.to_csv(f'{table_dir}/bosc_period_plot_df.csv', index=False)
stats_long_df.to_csv(f'{table_dir}/bosc_period_stats_long.csv', index=False)
stats_summary_df.to_csv(f'{table_dir}/bosc_period_stats_summary.csv', index=False)

print(time_resolved_df.shape, period_df.shape, stats_summary_df.shape)
stats_summary_df.head(12)


In [ ]:
from membandit.extras.bosc import print_pairwise_rpe_ttests_period_plot

rois_band_plot = ['ACC', 'HPC']
events_band_plot = {'cue_on': events['cue_on']}

print_pairwise_rpe_ttests_period_plot(
    period_plot_df=period_plot_df,
    rois=rois_band_plot,
    events=events_band_plot,
    bands=bands,
)

# --- period-averaged BOSC by band (inline; edit freely) ---
event_names = list(events_band_plot.keys())
band_order = list(bands.keys())
n_rpe = len(RPE_CATEGORY_ORDER)

fig, axes = plt.subplots(
    len(rois_band_plot),
    len(event_names),
    figsize=(6, 3.4 * len(rois_band_plot)),
    sharey=True,
    dpi=300,
    squeeze=False,
)
axes = np.asarray(axes)

for i, roi in enumerate(rois_band_plot):
    for j, event in enumerate(event_names):
        ax = axes[i, j]
        plot_df = period_plot_df[
            (period_plot_df['region'] == roi)
            & (period_plot_df['event'] == event)
            & (period_plot_df['band'].isin(band_order))
        ].copy()
        raw_df = period_df[
            (period_df['region'] == roi)
            & (period_df['event'] == event)
            & (period_df['band'].isin(band_order))
        ].copy()
        n_elecs = int(raw_df['unique_elec_id'].nunique())
        n_subjects = int(plot_df['participant'].nunique())

        if plot_df.empty:
            ax.set_visible(False)
            continue

        sns.barplot(
            data=plot_df,
            x='band',
            y='prop_detect',
            hue='rpe_cat',
            order=band_order,
            hue_order=RPE_CATEGORY_ORDER,
            palette=rpe_palette,
            errorbar='se',
            ax=ax,
        )
        sns.stripplot(
            data=plot_df,
            x='band',
            y='prop_detect',
            hue='rpe_cat',
            order=band_order,
            hue_order=RPE_CATEGORY_ORDER,
            dodge=True,
            palette=rpe_palette,
            alpha=0.7,
            size=4,
            ax=ax,
        )

        handles, labels = ax.get_legend_handles_labels()
        if len(handles) > 0:
            ax.legend(handles[:n_rpe], labels[:n_rpe], frameon=False, title='RPE bin')

        ax.set_title(f'{roi} | {event} | n_subj={n_subjects}, n_elecs={n_elecs}')
        ax.set_xlabel('Band')
        ax.set_ylabel('Burst prevalence')
        ax.grid(False)
        sns.despine(ax=ax, top=True, right=True)
        ax.tick_params(
            axis='both', which='major', bottom=True, left=True, direction='out', length=4
        )

fig.suptitle('Period-averaged BOSC prevalence by RPE bin and band', y=1.02)
fig.tight_layout()

period_band_fig = fig
period_band_fig.savefig(
    f'{figure_dir}/period_averaged_band_comparison_rpe_bins.pdf',
    dpi=300,
    bbox_inches='tight',
)
plt.show()


In [ ]:
f'{figure_dir}/period_averaged_band_comparison_rpe_bins.pdf'

In [ ]:
alignment_check = (
    period_df[[
        'participant',
        'event',
        'trial',
        'behavior_source',
        'behavior_trial_col',
        'rpe',
        'rpe_cat',
        'region',
        'band',
    ]]
    .drop_duplicates()
    .sort_values(['participant', 'event', 'trial'])
    .reset_index(drop=True)
)
alignment_check.to_csv(f'{table_dir}/bosc_alignment_check.csv', index=False)

band_check = (
    freq_df.assign(band=freq_df['frequency'].apply(assign_exact_band))
    .dropna(subset=['band'])
    .groupby('band')['frequency']
    .agg(['min', 'max', 'nunique'])
    .reset_index()
)
band_check.to_csv(f'{table_dir}/bosc_band_check.csv', index=False)

alignment_check.head(), band_check
